In [1]:
# 경고 메시지 무시
import warnings
warnings.filterwarnings(action='ignore') 

import pandas as pd
import numpy as np
import csv
import folium
import datetime
import seaborn as sns
import scipy as sp
import statsmodels.formula.api as smf
import networkx as nx
import missingno as msno
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_diabetes
import os
import sys
import urllib.request
import time
import json

from dotenv import load_dotenv
from folium.plugins import HeatMap 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler 
from dateutil.relativedelta import relativedelta
from sklearn.cluster import KMeans    ##  K-means 임포트
from sklearn.metrics import silhouette_score
from yellowbrick.cluster import KElbowVisualizer
from scipy.cluster.hierarchy import dendrogram, linkage
from mpl_toolkits.mplot3d import Axes3D
from operator import itemgetter
from PIL import Image
from collections import Counter
from wordcloud import WordCloud
import re


import matplotlib.pyplot as plt
plt.rc('font',family='D2CodingLigature Nerd Font')
# plt.rc('font', family='malgun gothic')
# plt.rcParams['axes.unicode_minus']=False  # '- 표시

## 인터랙티브 그래프(interactive graph)
마우스 움직임에 반응하며 실시간으로 해당 부분의 값이 표현되는 그래프

In [2]:
mpg = pd.read_csv('../../data/mpg.csv', encoding='utf-8')
mpg

,manufacturer,model,displ,year,cyl,trans,drv,cty,hwy,fl,category
0,audi,a4,1.8,1999,4,auto(l5),f,18,29,p,compact
1,audi,a4,1.8,1999,4,manual(m5),f,21,29,p,compact
2,audi,a4,2.0,2008,4,manual(m6),f,20,31,p,compact
3,audi,a4,2.0,2008,4,auto(av),f,21,30,p,compact
4,audi,a4,2.8,1999,6,auto(l5),f,16,26,p,compact
...,...,...,...,...,...,...,...,...,...,...,...
229,volkswagen,passat,2.0,2008,4,auto(s6),f,19,28,p,midsize
230,volkswagen,passat,2.0,2008,4,manual(m6),f,21,29,p,midsize
231,volkswagen,passat,2.8,1999,6,auto(l5),f,16,26,p,midsize
232,volkswagen,passat,2.8,1999,6,manual(m5),f,18,26,p,midsize


In [3]:
mpg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   manufacturer  234 non-null    object 
 1   model         234 non-null    object 
 2   displ         234 non-null    float64
 3   year          234 non-null    int64  
 4   cyl           234 non-null    int64  
 5   trans         234 non-null    object 
 6   drv           234 non-null    object 
 7   cty           234 non-null    int64  
 8   hwy           234 non-null    int64  
 9   fl            234 non-null    object 
 10  category      234 non-null    object 
dtypes: float64(1), int64(4), object(6)
memory usage: 20.2+ KB


In [4]:
import plotly.express as px

In [5]:
px.scatter(data_frame = mpg, x = 'cty', y = 'hwy', color = 'drv')

In [6]:
# 그래프를 변수에 할당하기
fig=px.scatter(data_frame = mpg, x = 'cty', y = 'hwy', color = 'drv')

# html로 저장하기
fig.write_html('../../data/scatter_plot.html')

In [7]:
# 자동차 종류별 빈도 구하기

df = mpg.groupby('category', as_index = False).agg(n = ('category', 'count'))
df

,category,n
0,2seater,5
1,compact,47
2,midsize,41
3,minivan,11
4,pickup,33
5,subcompact,35
6,suv,62


In [9]:
px.bar(data_frame = df, x = 'category', y = 'n', color = 'category')

In [11]:
fig2 = px.bar(data_frame = df, x = 'category', y = 'n', color = 'category')

fig2.write_html('../../data/bar_plot.html')

In [13]:
# 상자 그림
px.box(data_frame = mpg, x = 'drv', y = 'hwy', color= 'drv')

In [21]:
# seaborn에서 mpg 데이터셋 불러오기
mpg2 = sns.load_dataset('mpg')
mpg2

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino
...,...,...,...,...,...,...,...,...,...
393,27.0,4,140.0,86.0,2790,15.6,82,usa,ford mustang gl
394,44.0,4,97.0,52.0,2130,24.6,82,europe,vw pickup
395,32.0,4,135.0,84.0,2295,11.6,82,usa,dodge rampage
396,28.0,4,120.0,79.0,2625,18.6,82,usa,ford ranger


In [22]:
fig3 = px.box(mpg2, x='origin', y='mpg', color='origin')

In [27]:
# 각 트레이스에 대해 hovertemplate을 한글로 표시되게 설정 (이상치만 변경된다)
for trace in fig3.data:
    q1 = np.percentile(trace.y, 25)  # 1사분위수
    q3 = np.percentile(trace.y, 75)  # 3사분위수
    median = np.median(trace.y)      # 중앙값
    upperfence = q3 + 1.5 * (q3 - q1)  # 상위 경계
    lowerfence = q1 - 1.5 * (q3 - q1)  # 하위 경계
    ymax = np.max(trace.y)           # 최대값
    ymin = np.min(trace.y)           # 최소값
    n = len(trace.y)                 # 값의 수

    # hovertemplate 업데이트
    trace.update(
        hovertemplate=
        f'<b>최대값:</b> {ymax}<br>' +
        f'<b>상위 경계:</b> {upperfence}<br>' +
        f'<b>Q3 (3사분위수):</b> {q3}<br>' +
        f'<b>Q2 (중앙값):</b> {median}<br>' +
        f'<b>Q1 (1사분위수):</b> {q1}<br>' +
        f'<b>하위 경계:</b> {lowerfence}<br>' +
        f'<b>최소값:</b> {ymin}<br>' +
        f'<b>값의 수:</b> {n}<br>' +
        '<b>값:</b> %{y}<extra></extra>',  # 값 표시 부분
        hoverlabel=dict(
            font=dict(family="D2CodingLigature Nerd Font", size=12)  # 한글 폰트 설정
        )
    )

# 그래프 출력
fig3.show()